# Scraping Komentar Instagram (Post & Reels)
Notebook ini digunakan untuk melakukan scraping data komentar dari postingan atau reels Instagram secara aman menggunakan library `instagrapi`.

In [1]:
import os
import time
import random
import pandas as pd
from tqdm import tqdm
from instagrapi import Client
from instagrapi.exceptions import ClientError
from datetime import datetime


## 1. Konfigurasi Akun & Target URL
> **Catatan Penting**: Sangat disarankan untuk menggunakan **akun cadangan (akun tumbal)**, bukan akun pribadi utama Anda, untuk menghindari risiko akun diblokir/dibatasi oleh Instagram.

In [ ]:
# Masukkan Kredensial Akun Instagram Anda
IG_USERNAME = "rifkirif2024"
IG_PASSWORD = "RIFKIig2024"

# Masukkan URL postingan atau reels Instagram yang ingin dicari komentarnya
target_ig_urls = [
    # CORETAX,
    # "https://www.instagram.com/p/DWYrb4PEwZ-/",
    # "https://www.instagram.com/reel/DXd-RSckWCU/",
    # "https://www.instagram.com/reel/DV2kPF7kg__/",
    # "https://www.instagram.com/reel/DVScFc0k7sm/",
    # "https://www.instagram.com/p/DM7H6ToJOLM/",
    # "https://www.instagram.com/p/DX6bTT-kT18",
    # "https://www.instagram.com/reel/DVvT5gZgTby/",
    # "https://www.instagram.com/reel/DSbVBsoEq8B/",
    # "https://www.instagram.com/p/DX2zbU9AS4x/",
    # "https://www.instagram.com/p/DXwKIHGkUCt/",
    # "https://www.instagram.com/p/DUFPyntkuvi/",
    # "https://www.instagram.com/p/DNCnMRdTWgC/",

    # BPJS KESEHATAN
    # "https://www.instagram.com/reel/DYfSoAMCLbk/",
    # "https://www.instagram.com/reel/DXrT9W6oHrv/",
    # "https://www.instagram.com/reel/DXecV8ticye/",
    # "https://www.instagram.com/reel/DXea5HrD9Wq/",
    # "https://www.instagram.com/reel/DXZJx57gM7s/",
    # "https://www.instagram.com/p/DXVo_2hj4t8/",
    # "https://www.instagram.com/p/DXOcqQxlKc0/",
    # "https://www.instagram.com/reel/DVsm41_E0K2/",
    # "https://www.instagram.com/reel/DVf7NZZkVyL/",
    # "https://www.instagram.com/reel/DNSZNEwSo7E/",

    # MBG
    # "https://www.instagram.com/p/DWVXhkrk__j/",
    # "https://www.instagram.com/p/DXgjNKaDO2r/",
    # "https://www.instagram.com/p/DYuQExygdrV/",
    # "https://www.instagram.com/p/DYrGNJOFIRc/",
    # "https://www.instagram.com/p/DYomtlKARQQ/",
    # "https://www.instagram.com/reel/DYWUUopJgXC/",
    # "https://www.instagram.com/p/DYRquOxifSj/",
    # "https://www.instagram.com/reel/DYKD0t3Jxtw/",
    # "https://www.instagram.com/reel/DYE4JXvJGqS/",
    # "https://www.instagram.com/reel/DYmu1e6PyWi/",
    # "https://www.instagram.com/reel/DT-S7QPicLA/",
    # "https://www.instagram.com/reel/DYWnJzzpojp/",
]

# Batas maksimal komentar yang diambil per postingan.
# Gunakan angka wajar (misal: 300 atau 500) untuk menghindari rate-limit.
# Set ke 0 jika ingin mengambil SEMUA komentar (risiko terblokir lebih tinggi).
MAX_COMMENTS_PER_POST = 0

## 2. Fungsi Scraping dengan Session Management & Jeda (Delay)
Fungsi ini akan menyimpan sesi login Anda ke file `instagram_session.json` agar pada eksekusi berikutnya tidak perlu melakukan login ulang dari awal (sangat membantu mencegah deteksi bot oleh Instagram).

In [33]:
def scrape_instagram_comments(username, password, urls, max_comments=500):
    cl = Client()
    session_file = "instagram_session.json"
    
    # 1. Proses Login (Memanfaatkan Session Cache)
    if os.path.exists(session_file):
        print(">>> Memuat session login dari cache...")
        try:
            cl.load_settings(session_file)
            cl.login(username, password)
            print("Login berhasil menggunakan session cache!\n")
        except Exception as e:
            print(f"Gagal login menggunakan session cache: {e}")
            print("Mencoba login ulang dengan username & password...")
            try:
                cl.login(username, password)
                cl.dump_settings(session_file)
                print("Login berhasil dan session baru disimpan!\n")
            except Exception as login_err:
                print(f"Gagal login: {login_err}")
                return pd.DataFrame()
    else:
        print(">>> Melakukan login baru...")
        try:
            cl.login(username, password)
            cl.dump_settings(session_file)
            print("Login berhasil dan session disimpan!\n")
        except Exception as e:
            print(f"Gagal login: {e}")
            return pd.DataFrame()

    # 2. Proses Scraping Komentar
    all_comments = []
    for url in urls:
        print(f"Scraping URL: {url}")
        try:
            # Mendapatkan ID internal media dari URL (mendukung /p/ dan /reel/)
            media_pk = cl.media_pk_from_url(url)
            
            # Mengambil komentar
            comments = cl.media_comments(media_pk, amount=max_comments)
            
            for c in comments:
                all_comments.append({
                    "id": str(c.pk),
                    "timestamp": c.created_at_utc.isoformat() if c.created_at_utc else None,
                    "likesCount": c.like_count,
                    "postUrl": url,
                    "commentUrl": f"{url.split('?')[0]}?comment_id={c.pk}",
                    "source_file": "scraping_instagram",
                    "ownerUsername": c.user.username if c.user else None,
                    "text": c.text
                })
            
            # Berikan jeda acak 5-10 detik antara setiap postingan agar aman dari blokir
            delay = random.uniform(5, 10)
            print(f"-> Berhasil mengambil {len(comments)} komentar. Jeda {delay:.2f} detik...")
            time.sleep(delay)
            
        except Exception as e:
            print(f"-> Gagal memproses URL {url}: {e}")
            time.sleep(5)  # Jeda sejenak jika terjadi error
            
    return pd.DataFrame(all_comments)

## 3. Jalankan Scraping & Simpan ke CSV
Format CSV akan disesuaikan dengan format dataset pelabelan Anda (`id, timestamp, likesCount, postUrl, commentUrl, source_file, ownerUsername, text`).

In [34]:
df_ig = scrape_instagram_comments(IG_USERNAME, IG_PASSWORD, target_ig_urls, max_comments=MAX_COMMENTS_PER_POST)

if not df_ig.empty:
    # Bersihkan baris yang tidak memiliki teks komentar
    df_ig = df_ig.dropna(subset=['text'])
    
    # Membuat nama file otomatis dengan timestamp saat ini (TahunBulanTanggal_JamMenitDetik)
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"dataset_instagram_{timestamp}.csv"
    
    # Menyimpan file dengan encoding utf-8-sig agar karakter emoji tersimpan dengan baik
    df_ig.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print("✅ SCRAPING INSTAGRAM SELESAI!")
    print(f"Total Komentar Terambil: {len(df_ig)}")
    print(f"Data berhasil disimpan ke file '{output_filename}'")
    print("="*50)
else:
    print("\n❌ Scraping gagal atau tidak ada komentar yang didapatkan.")


>>> Memuat session login dari cache...
Login berhasil menggunakan session cache!

Scraping URL: https://www.instagram.com/reel/DYWnJzzpojp/
-> Berhasil mengambil 234 komentar. Jeda 8.65 detik...

✅ SCRAPING INSTAGRAM SELESAI!
Total Komentar Terambil: 234
Data berhasil disimpan ke file 'dataset_instagram_20260525_150259.csv'
